# Week 3 — Working with External Libraries

Covers: instantiating and calling SDK-style clients (using a mock vector DB client, since
`chromadb` isn't part of the core runtime — see `requirements.txt`), and basic `sqlite3`
usage for the SQL-RAG capstone option.

## 1. The SDK client pattern — demonstrated with a mock vector store

In [ ]:
import numpy as np

class MockVectorCollection:
    """Same shape as a real chromadb/qdrant collection client:
    instantiate it, .add() documents, .query() for nearest neighbours."""

    def __init__(self, name):
        self.name = name
        self._ids = []
        self._vectors = []
        self._documents = []

    def add(self, ids, embeddings, documents):
        self._ids.extend(ids)
        self._vectors.extend(embeddings)
        self._documents.extend(documents)

    def query(self, query_embedding, top_k=2):
        query_embedding = np.array(query_embedding)
        sims = []
        for i, vec in enumerate(self._vectors):
            vec = np.array(vec)
            sim = np.dot(query_embedding, vec) / (np.linalg.norm(query_embedding) * np.linalg.norm(vec))
            sims.append((sim, self._ids[i], self._documents[i]))
        sims.sort(reverse=True)
        return sims[:top_k]

# Usage — this is (almost) exactly how you'd call the real chromadb client
collection = MockVectorCollection(name="policy_clauses")
collection.add(
    ids=["clause_4_1", "clause_4_2"],
    embeddings=[[0.1, 0.9, 0.0], [0.15, 0.85, 0.05]],
    documents=["Wear and tear is excluded.", "Pre-existing damage is excluded."],
)

results = collection.query(query_embedding=[0.12, 0.88, 0.02], top_k=2)
for score, doc_id, text in results:
    print(f"{score:.4f}  {doc_id}  ->  {text}")

Swapping this for the real library later is mostly a constructor change:

```python
import chromadb
client = chromadb.Client()
collection = client.create_collection(name="policy_clauses")
collection.add(ids=[...], embeddings=[...], documents=[...])
collection.query(query_embeddings=[...], n_results=2)
```

Same shape: instantiate a client, `.add()`, `.query()`. That's the pattern this section teaches.

## 2. `sqlite3` — for the SQL-RAG / structured-lookup capstone path

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")   # in-memory DB, gone when the connection closes
cur = conn.cursor()

cur.execute('''
    CREATE TABLE claims (
        claim_id TEXT PRIMARY KEY,
        policy_id TEXT,
        status TEXT,
        amount INTEGER
    )
''')

rows = [
    ("C-1001", "P-55", "approved", 15000),
    ("C-1002", "P-12", "rejected", 0),
    ("C-1003", "P-55", "approved", 8000),
]
cur.executemany("INSERT INTO claims VALUES (?, ?, ?, ?)", rows)
conn.commit()

# A natural-language question like "how much has policy P-55 been paid?"
# becomes exactly this kind of parameterised query in a SQL-RAG agent.
cur.execute("SELECT SUM(amount) FROM claims WHERE policy_id = ? AND status = 'approved'", ("P-55",))
total = cur.fetchone()[0]
print("Total approved for P-55:", total)

conn.close()

Note the **parameterised query** (`?` placeholder, not an f-string building SQL directly).
This isn't a style preference — building SQL by string-formatting untrusted input is exactly
the SQL-injection risk that Week 5's guardrails module covers for LLM-driven queries.